# Who is who? Master data management, step by step

The same customer often appears in several systems — spelt differently, with dates in different formats and addresses that disagree. This notebook walks through how master data management (MDM) finds the matches, builds one trusted **golden record** per customer, and groups customers into **households**.

It is the Python twin of the interactive playground on my website (built in TypeScript). Both read the same sample data and produce identical results.

> A simplified illustration with fictional data — the concepts, not any employer's algorithm.

In [1]:
import pandas as pd

from mdm import (FIELDS, cluster, households, load_sample, run, score_all,
                 standardise_address, survive)

pd.set_option("display.max_colwidth", 60)

## Step 1 — Messy source data

Ten records from three systems: policy, claims and web sign-ups. Spot the spelling variants (*Smyth*, *Patell*), the mixed date formats and the abbreviations (*Oak Ln*, *Mill St*).

In [2]:
records, defaults = load_sample()
pd.DataFrame(records).set_index("id")

,source,name,dob,address,email,updated
id,,,,,,
P1,Policy,Jonathan Smith,1980-03-12,"14 Oak Lane, Leeds, LS1 4AB",jon.smith@example.com,2016-01-10
C1,Claims,Jon Smith,12/03/1980,"14 Oak Ln, Leeds LS1 4AB",,2017-06-02
W1,Web,J. Smyth,12-03-1980,14 Oak Lane LS14AB,jon.smith@example.com,2017-09-15
P2,Policy,Sarah Smith,1982-09-05,"14 Oak Lane, Leeds, LS1 4AB",sarah.smith@example.com,2016-01-10
W2,Web,Sarah Smith,05/09/1982,"22 Elm Road, Leeds, LS2 7QT",sarah.smith@example.com,2017-11-20
P3,Policy,Priya Patel,1975-07-21,"3 Mill Street, York, YO1 6AA",priya.patel@example.com,2015-04-18
C2,Claims,Priya Patell,21/07/1975,"3 Mill St, York YO1 6AA",,2016-08-30
P4,Policy,Ravi Patel,1974-02-14,"3 Mill Street, York, YO1 6AA",ravi.patel@example.com,2015-04-18
W3,Web,Daniel Evans,30/11/1990,"8 Station Road, Bath, BA1 1AA",dan.evans@example.com,2017-02-11


## Step 2 — Matching

Every pair of records is compared field by field and given a similarity between 0 and 1. The field scores are combined using **weights** into a match score out of 100. Fields missing on either side (often email) don't count.

- **85+** — merge automatically
- **70–84** — send to a person for review
- **below 70** — keep apart

In [3]:
weights, thresholds = defaults["weights"], defaults["thresholds"]
print("Weights:", weights, "| Thresholds:", thresholds)

pairs = score_all(records, weights, thresholds)
pd.DataFrame([
    {"pair": f"{p['a']['id']} {p['a']['name']} ↔ {p['b']['id']} {p['b']['name']}",
     **{f: None if p['scores'][f] is None else round(p['scores'][f], 2) for f in FIELDS},
     "score": p["score"], "decision": p["decision"]}
    for p in pairs if p["score"] >= 40
])

Weights: {'name': 35, 'dob': 30, 'address': 20, 'email': 15} | Thresholds: {'review': 70, 'auto': 85}


,pair,name,dob,address,email,score,decision
0,P3 Priya Patel ↔ C2 Priya Patell,0.98,1.0,1.00,NaN,99,match
1,P1 Jonathan Smith ↔ C1 Jon Smith,0.94,1.0,1.00,NaN,98,match
2,P1 Jonathan Smith ↔ W1 J. Smyth,0.88,1.0,1.00,1.0,96,match
3,C1 Jon Smith ↔ W1 J. Smyth,0.88,1.0,1.00,NaN,95,match
4,P2 Sarah Smith ↔ W2 Sarah Smith,1.00,1.0,0.14,1.0,83,review
5,W3 Daniel Evans ↔ C3 Danielle Evans,0.98,0.5,1.00,NaN,82,review
6,C2 Priya Patell ↔ P4 Ravi Patel,0.83,0.0,1.00,NaN,58,no-match
7,P3 Priya Patel ↔ P4 Ravi Patel,0.85,0.0,1.00,0.0,50,no-match
8,P1 Jonathan Smith ↔ P2 Sarah Smith,0.82,0.0,1.00,0.0,49,no-match
9,C1 Jon Smith ↔ P2 Sarah Smith,0.60,0.0,1.00,NaN,48,no-match


## Step 3 — Golden records (survivorship)

Automatically matched records merge into one trusted customer. **Survivorship rules** decide which source wins each field: here the most complete name, the most recent address, date of birth from the most trusted source (Policy, then Claims, then Web), and the most recent email.

In [4]:
rules = defaults["survivorship"]
golden = [survive(group, rules) for group in cluster(records, pairs)]
pd.DataFrame([
    {"customer": g["id"],
     "name": g["values"]["name"]["value"],
     "address": standardise_address(g["values"]["address"]["value"]),
     "address from": g["values"]["address"]["from"],
     "email": g["values"]["email"]["value"] or "—"}
    for g in golden
])

,customer,name,address,address from,email
0,P1+C1+W1,Jonathan Smith,"14 Oak Lane, LS1 4AB",Web,jon.smith@example.com
1,P2,Sarah Smith,"14 Oak Lane Leeds, LS1 4AB",Policy,sarah.smith@example.com
2,W2,Sarah Smith,"22 Elm Road Leeds, LS2 7QT",Web,sarah.smith@example.com
3,P3+C2,Priya Patel,"3 Mill Street York, YO1 6AA",Claims,priya.patel@example.com
4,P4,Ravi Patel,"3 Mill Street York, YO1 6AA",Policy,ravi.patel@example.com
5,W3,Daniel Evans,"8 Station Road Bath, BA1 1AA",Web,dan.evans@example.com
6,C3,Danielle Evans,"8 Station Road Bath, BA1 1AA",Claims,—


## Step 4 — Households

Customers sharing an address — the same house number and postcode — form a household.

In [5]:
pd.DataFrame([
    {"household": h["key"],
     "members": ", ".join(m["values"]["name"]["value"] for m in h["members"])}
    for h in households(golden)
])

,household,members
0,14 LS14AB,"Jonathan Smith, Sarah Smith"
1,22 LS27QT,Sarah Smith
2,3 YO16AA,"Priya Patel, Ravi Patel"
3,8 BA11AA,"Daniel Evans, Danielle Evans"


## The trade-off — set the bar too low and two people become one

Daniel and Danielle Evans share an address and have birth years one apart. At the default threshold they go to review. Lower the automatic-merge threshold to 80 and they are wrongly merged — the false-match risk every MDM programme has to manage.

In [6]:
for auto in (85, 80):
    result = run(records, weights, {**thresholds, "auto": auto}, rules)
    merged = [g["id"] for g in result["golden"] if "+" in g["id"]]
    print(f"auto-merge at {auto}: {result['summary']}  merged: {merged}")

auto-merge at 85: {'records': 10, 'customers': 7, 'households': 4, 'review': 2}  merged: ['P1+C1+W1', 'P3+C2']
auto-merge at 80: {'records': 10, 'customers': 5, 'households': 4, 'review': 0}  merged: ['P1+C1+W1', 'P2+W2', 'P3+C2', 'W3+C3']


## Parity with the TypeScript playground

The website's interactive playground runs the same logic in TypeScript. The results below are committed to the repository, and **the website build fails if the TypeScript engine ever disagrees** — so the two implementations can't drift apart.

In [7]:
import json
from mdm import ROOT

committed = json.loads((ROOT / "src" / "data" / "mdm-parity.json").read_text(encoding="utf-8"))
print("Python results match the committed parity file:",
      committed == run(records, weights, thresholds, rules))

Python results match the committed parity file: True
